# Shor's algorithm — factor 15

Shor reduces factoring $N$ to **finding the period** $r$ of $f(x) = a^x \bmod N$.

For $N=15$ and $a=7$ the sequence is $7,4,13,1,7,\ldots$ so $r=4$. Then

$$
\gcd(a^{r/2}\pm 1,\ N) = \gcd(48\pm 1,\ 15) = (3,5).
$$

The quantum piece is phase estimation of the unitary $|k\rangle\mapsto|a k \bmod N\rangle$.
This notebook rebuilds that circuit locally; it does not call Grover, QAOA, or `shor.py`.

In [ ]:
import qiskit as qk
import qiskit_aer as qka


class std:
    import math
    import fractions


N = 15
A = 7
COUNTING = 8
print("gcd(a, N) must be 1:", std.math.gcd(A, N))
print("classical orbit:", [pow(A, x, N) for x in range(1, 9)])

## Modular multiply $a \cdot k \bmod 15$

On four qubits the allowed bases $\{2,4,7,8,11,13\}$ collapse to a few
SWAPs and X gates. Controlled-$U^{2^k}$ is that block repeated $2^k$
times and wrapped in a control.

In [ ]:
def multiply_mod15(a):
    u = qk.QuantumCircuit(4, name=f"*{a} mod 15")
    if a in (2, 13):
        u.swap(2, 3); u.swap(1, 2); u.swap(0, 1)
    if a in (7, 8):
        u.swap(0, 1); u.swap(1, 2); u.swap(2, 3)
    if a in (4, 11):
        u.swap(1, 3); u.swap(0, 2)
    if a in (7, 11, 13):
        for q in range(4):
            u.x(q)
    return u


def controlled_power(a, exponent):
    body = qk.QuantumCircuit(4)
    for _ in range(exponent):
        body.compose(multiply_mod15(a), inplace=True)
    gate = body.to_gate()
    gate.name = f"{a}^{exponent} mod 15"
    return gate.control(1)

## Inverse QFT, written gate by gate

In [ ]:
def inverse_qft(n):
    iqft = qk.QuantumCircuit(n, name="IQFT")
    for i in range(n // 2):
        iqft.swap(i, n - 1 - i)
    for j in range(n):
        for k in range(j):
            iqft.cp(-std.math.pi / 2 ** (j - k), k, j)
        iqft.h(j)
    return iqft


print(inverse_qft(4).draw())

## Phase-estimation circuit and a handful of shots

In [ ]:
def shor_circuit(a, n_count):
    phase = qk.QuantumRegister(n_count, "phase")
    work = qk.QuantumRegister(4, "work")
    meas = qk.ClassicalRegister(n_count, "m")
    qc = qk.QuantumCircuit(phase, work, meas)
    qc.h(phase)
    qc.x(work[0])
    for k, q in enumerate(phase):
        qc.append(controlled_power(a, 2 ** k), [q, *work])
    qc.append(inverse_qft(n_count), phase)
    qc.measure(phase, meas)
    return qc


qc = shor_circuit(A, COUNTING)
sim = qka.AerSimulator()
counts = sim.run(qk.transpile(qc, sim), shots=128).result().get_counts()
for bits, n in sorted(counts.items(), key=lambda kv: -kv[1])[:8]:
    value = int(bits, 2)
    print(f"{bits}  {value}/{2**COUNTING} = {value / 2**COUNTING:.4f}  x{n}")

## Continued fractions recover $r$

A measured integer $s$ estimates the phase $s/2^n \approx k/r$. The
convergent of that fraction whose denominator is even and less than $N$
is the period we need.

In [ ]:
def period_from_measurement(value, n_count, n):
    if value == 0:
        return None
    frac = std.fractions.Fraction(value, 2 ** n_count).limit_denominator(n)
    return frac.denominator if 0 < frac.denominator <= n else None


def factors(a, r, n):
    if r % 2:
        return None
    x = pow(a, r // 2, n)
    if x in (1, n - 1):
        return None
    p, q = std.math.gcd(x - 1, n), std.math.gcd(x + 1, n)
    if 1 < p < n:
        return p, n // p
    if 1 < q < n:
        return q, n // q
    return None


found = None
for bits, _n in sorted(counts.items(), key=lambda kv: -kv[1]):
    value = int(bits, 2)
    r = period_from_measurement(value, COUNTING, N)
    if r is None:
        continue
    pair = factors(A, r, N)
    print(f"value={value:3d}  r={r}  factors={pair}")
    if pair:
        found = pair
        break
print("15 =", " x ".join(map(str, found)) if found else "try another shot record")